# Get a Google Maps link

This notebook generates a Google Maps link in the same format as the Apps Script handler of the [Interest Form](https://docs.google.com/forms/d/e/1FAIpQLSesAn9rI47UeUScOmhiPRhFXnUkpmC4WA2abuI31mviXomAQw/viewform).

This is intended to populate the Google Maps link for a manually created Trello card.

This is needed because the Google Maps link is parsed to get the longitude and latitude for the address when creating the node GeoJSON file using the [trello-to-geojson](https://github.com/tucsonmesh/trello-to-geojson) script.

In [1]:
from urllib.parse import quote_plus

from IPython.display import display, HTML
import ipywidgets as widgets
import requests

In [2]:
def get_address_geojson(address: str) -> dict:
    """Returns a GeoJSON object for the address"""
    # See https://nominatim.org/release-docs/develop/api/Search/
    endpoint_url = "https://nominatim.openstreetmap.org/search"
    params = {
        "street": address,
        "city": "Tucson",
        "state": "AZ",
        "country": "United States",
        "format": "geojson",
        # Include address details because we eventually want to make our own address format instead of the default
        "addressdetails": 1,
    }
    # You must specify a non-default user agent
    headers = {"User-Agent": "tucson-mesh-maps-link/0.1.0"}
    resp = requests.get(endpoint_url, params=params, headers=headers)
    return resp.json()

In [3]:
def get_formatted_address(feature_geojson: dict) -> str:
    """Returns a consistently formatted address"""
    return (
        f"{feature_geojson['properties']['address']['house_number']} {feature_geojson['properties']['address']['road']}, "
        f"{feature_geojson['properties']['address']['city']}, {feature_geojson['properties']['address']['state']} "
        f"{feature_geojson['properties']['address']['postcode']}"
    )

In [4]:
def get_maps_url(feature_geojson: dict) -> str:
    """Returns a link for the address on Google Maps"""
    lat = feature_geojson["geometry"]["coordinates"][1]
    lng = feature_geojson["geometry"]["coordinates"][0]
    return (
        "https://www.google.com/maps/place/"
        f"{quote_plus(get_formatted_address(feature_geojson))}/"
        f"@{lat},{lng}"
    )

In [5]:
def get_maps_link(feature_geojson: dict) -> str:
    """Returns HTML for a link to Google Maps"""
    maps_url = get_maps_url(feature_geojson)

    return f'<a href="{maps_url}">See on Google Maps</a>'

In [6]:
address_input = widgets.Text(
    value="",
    placeholder="Address",
    disabled=False   
)
submit_button = widgets.Button(description="Get Address")
output = widgets.Output()

display(address_input, submit_button, output)

def on_button_clicked(b):
    feature = get_address_geojson(address_input.value)["features"][0]
    formatted_address = get_formatted_address(feature)
    maps_link = get_maps_link(feature)
    output.clear_output()
    with output:
        display(HTML(f"<p>{formatted_address}</p>"))
        display(HTML(f"<p>{maps_link}</p>"))

submit_button.on_click(on_button_clicked)

Text(value='', placeholder='Address')

Button(description='Get Address', style=ButtonStyle())

Output()